# MCP con LangChain y Ollama

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/2-mcp-con-langchain-y-ollama.ipynb)

En este segundo notebook reutilizaremos los mismos dos casos del notebook anterior, pero ahora exponiendo las capacidades vía MCP. La meta es que el estudiante vea el cambio arquitectónico con el mismo problema de negocio: primero conectaremos un cliente MCP explícito para entender el protocolo y luego cargaremos esas herramientas dentro de un agente que las consume a través de LangChain.

### Referencias
- [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://pypi.org/project/langchain-mcp-adapters/)
- [Open-Meteo API](https://open-meteo.com/)
- [Ollama](https://ollama.com/)


In [1]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
!test '{IN_COLAB}' = 'True' && pip install "mcp>=1.24.0,<2.0.0" "langchain-mcp-adapters>=0.3.0,<0.4.0" langchain langchain-core langchain-ollama langgraph httpx ollama colab-xterm

# En local, instala/actualiza con: pip install -r requirements.txt

### Cargando a Ollama

Usaremos el mismo modelo local de la lección anterior para que el contraste se concentre en la arquitectura y no en cambiar de modelo.


In [ ]:
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


## Atención

En Colab, si el servidor de Ollama no está levantado todavía, inícialo en la terminal embebida. Si corres en local y ya tienes `ollama serve`, basta con continuar.

In [ ]:
%load_ext colabxterm
%xterm


Mantendremos `llama3.2:3b` para la demostración.


In [ ]:
!ollama pull llama3.2:3b


## Paso 1: revisamos dos servidores MCP pequeños

En este notebook **no** vamos a ocultar la implementación dentro de cadenas largas. Los servidores MCP viven como archivos Python normales en el repositorio para que puedas leerlos, editarlos y depurarlos como cualquier otro módulo.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / 'Sesion6' / 'mcp_servers').exists():
    SERVERS_DIR = REPO_ROOT / 'Sesion6' / 'mcp_servers'
else:
    SERVERS_DIR = REPO_ROOT / 'mcp_servers'

calculator_server = SERVERS_DIR / 'calculator_mcp_server.py'
weather_server = SERVERS_DIR / 'weather_mcp_server.py'

assert calculator_server.exists(), f'No existe: {calculator_server}'
assert weather_server.exists(), f'No existe: {weather_server}'

SERVERS_DIR

In [ ]:
print(f'Servidor calculadora: {calculator_server}')
print('-' * 80)
print(calculator_server.read_text(encoding='utf-8'))

In [ ]:
print(f'Servidor clima: {weather_server}')
print('-' * 80)
print(weather_server.read_text(encoding='utf-8'))

## Inicio y apagado de servidores MCP (claro para clase)

Tienes dos formas válidas de ejecutar estos servidores:

1. Modo automático (recomendado para este notebook):
   Los ejemplos con `stdio_client` y `MultiServerMCPClient` levantan y cierran los procesos automáticamente durante la ejecución de cada llamada.
2. Modo manual (útil para observar procesos):
   Puedes iniciarlos en una terminal aparte y detenerlos con `Ctrl+C`.

Si usas modo manual, **apaga los procesos al terminar** para evitar sesiones colgadas.

In [ ]:
import sys

manual_start_commands = [
    f"{sys.executable} {calculator_server}",
    f"{sys.executable} {weather_server}",
]

print('Comandos para modo manual (ejecutar en terminales separadas):')
for cmd in manual_start_commands:
    print(f'- {cmd}')

print('\nPara detener cada servidor manual: Ctrl+C en su terminal.')

## Paso 2: usamos un cliente MCP explícito

Antes de conectarlo a un agente, vale la pena mirar el protocolo casi sin ayudas. Inicializamos una sesión, listamos herramientas y ejecutamos una de ellas. Así se vuelve más claro qué parte pertenece al servidor y qué parte pertenece al host o cliente.

Nota: en este ejemplo con transporte `stdio`, el cliente lanza el proceso del servidor automáticamente; no necesitas otra terminal para este paso.

In [4]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def probar_calculadora_mcp():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(calculator_server)],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            result = await session.call_tool('calculadora', {'expression': '(125 * 17) + 938'})
            return tools, result

tools_info, calc_result = await probar_calculadora_mcp()
tools_info, calc_result


(ListToolsResult(meta=None, nextCursor=None, tools=[Tool(name='calculadora', title=None, description='Evalúa una expresión aritmética segura con +, -, *, /, %, ** y paréntesis.', inputSchema={'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculadoraArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'calculadoraOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None)]),
 CallToolResult(meta=None, content=[TextContent(type='text', text='Resultado exacto: 3063', annotations=None, meta=None)], structuredContent={'result': 'Resultado exacto: 3063'}, isError=False))

In [5]:
if hasattr(calc_result, 'structured_content'):
    calc_result.structured_content
else:
    calc_result.content


Esa llamada ya es MCP real: hay un servidor, un transporte, un cliente y una invocación de tool definida por un contrato común. Todavía no hay agente, pero ya existe interoperabilidad.

## Paso 3: capa híbrida con un agente consumidor de herramientas MCP

Ahora sí reintroducimos la experiencia de agente. La diferencia es que las herramientas ya no están embebidas en este notebook: vienen publicadas por servidores MCP y el agente las descubre a través del adaptador de LangChain.

### Cómo decide el agente usar tools

Aunque el agente conozca tools, **no siempre las invoca**. En modo autónomo, el modelo decide si responde con conocimiento interno o si llama una herramienta.

Esto no significa que MCP esté roto: significa que la política de decisión del agente/modelo eligió no usar tool en ese turno.

Para enseñar esto con claridad, usaremos tres estrategias:

1. Autónomo: el agente decide libremente.
2. Guiado: damos una preferencia de tool y opcionalmente reintentamos con fallback.
3. Determinístico: ejecutamos la tool explícitamente para garantizar demostración reproducible en clase.

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

client = MultiServerMCPClient(
    {
        'calculadora': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(calculator_server)],
        },
        'clima': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(weather_server)],
        },
    }
)

mcp_tools = await client.get_tools()
agent_mcp = create_react_agent(model=llm, tools=mcp_tools)

def _tool_message_used(messages) -> bool:
    return any(getattr(msg, 'type', '') == 'tool' for msg in messages)

async def ejecutar_tool_mcp(tool_name: str, payload: dict) -> str:
    tool = next((t for t in mcp_tools if t.name == tool_name), None)
    if tool is None:
        raise ValueError(f'Herramienta no encontrada: {tool_name}')
    tool_result = await tool.ainvoke(payload)
    return str(tool_result)

async def preguntar_via_mcp(
    question: str,
    force_tool_name: str | None = None,
    force_tool_payload: dict | None = None,
):
    # Ruta determinística para clase: ejecuta explícitamente la herramienta MCP
    # y luego usa el LLM solo para redactar la respuesta final en español.
    if force_tool_name:
        payload = force_tool_payload or {}
        tool_output = await ejecutar_tool_mcp(force_tool_name, payload)
        synthesis = llm.invoke([
            ('system', 'Eres un asistente útil. Responde en español, de forma breve, y SOLO usando la salida de herramienta proporcionada.'),
            ('user', f"Pregunta original: {question}\n\nSalida de {force_tool_name}: {tool_output}\n\nRedacta la respuesta final en una oración."),
        ])
        trace = {
            'tool_forced': True,
            'used_tool': True,
            'forced_tool_name': force_tool_name,
            'forced_tool_output': tool_output,
        }
        return synthesis.content, trace

    # Ruta libre (agente decide si llama o no herramientas).
    result = await agent_mcp.ainvoke({'messages': [('user', question)]})
    messages = result['messages']
    used_tool = _tool_message_used(messages)
    final_message = messages[-1]
    trace = {**result, 'tool_forced': False, 'used_tool': used_tool}
    return final_message.content, trace

### Comparación rápida: modo autónomo (sin forzar tool)

Este ejemplo muestra el comportamiento real de un agente autónomo: puede usar tools o no, según cómo interprete la pregunta y sus instrucciones.

## Caso 1: la calculadora ahora viaja por MCP


In [7]:
respuesta_autonoma, traza_autonoma = await preguntar_via_mcp(
    '¿Cuál es el clima actual en Cali, Colombia? Si necesitas datos en tiempo real, usa herramientas.'
)
print(respuesta_autonoma)
print('¿El agente usó tool en modo autónomo?:', traza_autonoma.get('used_tool'))

Lo siento, pero no tengo acceso a datos en tiempo real. Sin embargo, puedo proporcionarte información sobre el clima promedio y las condiciones climáticas actuales en Cali, Colombia, según los datos disponibles hasta mi última actualización en diciembre de 2023.

Cali, Colombia tiene un clima tropical con dos estaciones: seca (diciembre a marzo) y lluviosa (abril a noviembre). Durante la estación seca, las temperaturas son más bajas, con promedios diurnos alrededor de 28°C y nocturnas alrededor de 18°C. La humedad es generalmente baja durante este período.

En cuanto a las condiciones climáticas actuales, según los datos disponibles hasta mi última actualización, Cali experimentó una temperatura promedio de 27°C en diciembre de 2023. Sin embargo, te recomiendo consultar fuentes más recientes y confiables para obtener información actualizada sobre el clima en Cali.

Algunas herramientas que puedes utilizar para obtener información actualizada sobre el clima incluyen:

*   OpenWeatherMap

In [8]:
respuesta_calculo_mcp, traza_calculo_mcp = await preguntar_via_mcp(
    '¿Cuánto es (125 * 17) + 938? Responde en español y menciona el resultado final.',
    force_tool_name='calculadora',
    force_tool_payload={'expression': '(125 * 17) + 938'},
)
print(respuesta_calculo_mcp)
if traza_calculo_mcp.get('tool_forced'):
    print('Nota: se forzó el uso de la herramienta MCP para asegurar la demostración.')

El resultado de la operación es 3063.
Nota: se forzó el uso de la herramienta MCP para asegurar la demostración.


## Caso 2: clima actual consumido como tool MCP


### Interpretación para estudiantes

- Si `used_tool=True`, hubo una llamada MCP dentro del ciclo del agente.
- Si `used_tool=False`, el modelo respondió directo sin herramienta (comportamiento válido en modo autónomo).

Para evaluaciones o demos donde necesites evidencia obligatoria de MCP, usa modo determinístico o guiado con fallback.

In [9]:
respuesta_clima_mcp, traza_clima_mcp = await preguntar_via_mcp(
    'Consulta el clima actual de Cali, Colombia, y resume la información más importante en una oración.',
    force_tool_name='clima_actual',
    force_tool_payload={'city': 'Cali, Colombia'},
)
print(respuesta_clima_mcp)
if traza_clima_mcp.get('tool_forced'):
    print('Nota: se forzó el uso de la herramienta MCP para evitar respuestas sin consulta en tiempo real.')

El clima actual en Cali, Colombia es soleado con una temperatura de 27.4°C y un viento suave de 0.6 km/h.
Nota: se forzó el uso de la herramienta MCP para evitar respuestas sin consulta en tiempo real.


In [10]:
print('Herramientas MCP disponibles:', [tool.name for tool in mcp_tools])
print('¿Se usó herramienta en modo autónomo?:', traza_autonoma.get('used_tool'))
print('¿Se usó herramienta en calculadora?:', traza_calculo_mcp.get('used_tool', not traza_calculo_mcp.get('tool_forced', False)))
print('¿Se usó herramienta en clima?:', traza_clima_mcp.get('used_tool', not traza_clima_mcp.get('tool_forced', False)))
print('¿Ruta determinística en calculadora?:', traza_calculo_mcp.get('tool_forced'))
print('¿Ruta determinística en clima?:', traza_clima_mcp.get('tool_forced'))

Herramientas MCP disponibles: ['calculadora', 'clima_actual']
¿Se usó herramienta en modo autónomo?: True
¿Se usó herramienta en calculadora?: True
¿Se usó herramienta en clima?: True
¿Ruta determinística en calculadora?: True
¿Ruta determinística en clima?: True


## Conclusiones

- En el notebook anterior ya teníamos herramientas útiles; aquí esas capacidades se transformaron en servicios interoperables.
- MCP separa mejor responsabilidades: el servidor expone capacidades y el host decide cómo consumirlas.
- La capa híbrida con LangChain ayuda a enseñar el flujo sin esconder el protocolo, mientras que la sección con `ClientSession` deja visible la mecánica básica de MCP.
